# Full concatenated pipeline with automatic ALI demixing

This cluster notebook preserves the existing concatenation, patterned-illumination preprocessing, and rigid NoRMCorre workflow, then replaces manual ROI drawing and VolPy with:

1. all-pixel motion-artifact projection;
2. a 50-ms pixelwise median residual for positive-event localization;
3. automatic full-session Activity Localization Imaging (ALI); and
4. an ALI-native result bundle for the local Dash viewer.

No manual cell ROI is required. SLM masks from `ROIs.xaml` constrain only the search area.


In [ ]:
import paramiko
import time
import re
import os
import glob
from pathlib import Path


# Define paths

Select the local mounted dataset folder. Session `.raw` files are concatenated under `output_concat/`, the first 500 frames of each session are omitted, and rigid NoRMCorre runs on the concatenated movie before ALI.


In [ ]:
# Local main folder
#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce47/2026-01-28_pAce47/PX/post/Awake' # 4X4
#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce45/2026-01-18_pAce45/PX/post/Awake' # 4X4, start from 3C next week
#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce38/2025-11-26_pAce38/PX/post/Awake' # 4X4
#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce21/2025-08-06_pAce21_PR/Awake' # 2X2, redo step 3
#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce28/2025-08-11-pAce28/Awake' # 2X2

#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce46/2026-02-22_pAce46/PR/post/Awake'
#main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce50/2025-03-17_pAce50_PRL/Awake'
main_folder_local = '/Volumes/adam-lab/Adam-Lab-Shared/Data/renana_malka/pAce54/2026-05-06_pAce54/PR/post/Awake'

# Cluster path mapping
main_folder = main_folder_local.replace('/Volumes/adam-lab', '/ems/elsc-labs/adam-y')

output_concat_subdir = 'output_concat'
output_concat = os.path.join(main_folder, output_concat_subdir)
output_concat_local = output_concat.replace('/ems/elsc-labs/adam-y', '/Volumes/adam-lab')

# Concatenated dataset settings
concat_raw_name = 'Image_001_001.raw'
frames_to_truncate_per_session = 500  # applied BEFORE concatenation (per-session)

# Motion correction settings (run on concatenated raw)
mc_output_subdir = 'output_final'
mc_run_tag = 'concat_mc'  # set to None to write directly under output_final/
mc_niter_rig = 2
mc_factor = (1, 1)
# if main_folder_local contains 'pAce21', then use mc_factor = (2, 2) instead of (1, 1)
if 'pAce21' in main_folder_local or 'pAce28' in main_folder_local:
    mc_factor = (2, 2)
    print('Using mc_factor (2, 2) for pAce21/pAce28 dataset')
mc_skip_roi_detection = True

print('main_folder_local:', main_folder_local)
print('output_concat_local:', output_concat_local)


In [ ]:
# Define cluster login details
CLUSTER_HOST = 'loginserver.elsc.huji.ac.il'
USERNAME = 'qixin.yang'
SSH_KEY_PATH = os.path.expanduser('~/.ssh/id_rsa')  # optional; set to None to use SSH agent

# Pipeline workdir + python runner
PIPELINE_WORKDIR = '/ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline'
sh_script = '/ems/elsc-labs/adam-y/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/utils/run_python_job.sh'

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
kwargs = {}
if SSH_KEY_PATH and os.path.exists(SSH_KEY_PATH):
    kwargs['key_filename'] = SSH_KEY_PATH
ssh.connect(CLUSTER_HOST, username=USERNAME, **kwargs)
print('[INFO] Connected to cluster.')


In [ ]:
def run_command(command):
    """Runs a command on the cluster with a login shell"""
    full_command = f"bash -l -c '{command}'"
    stdin, stdout, stderr = ssh.exec_command(full_command)
    output = stdout.read().decode().strip()
    error = stderr.read().decode().strip()
    return output, error

def wait_for_jobs(job_ids, poll_interval=30):
    """Poll SLURM until all jobs complete. Returns True if all succeeded."""
    import time
    from IPython.display import clear_output

    if not job_ids:
        print('No jobs to wait for.')
        return True

    pending_jobs = set(job_ids)
    failed_jobs = []

    while pending_jobs:
        # Check job status
        job_list = ','.join(pending_jobs)
        command = f"sacct -j {job_list} --format=JobID,State,ExitCode -n -P"
        output, error = run_command(command)

        if error and 'Invalid job id' not in error:
            print(f"[WARNING] Error checking jobs: {error}")

        # Parse output
        completed = set()
        for line in output.strip().splitlines():
            if not line or '.' in line.split('|')[0]:  # Skip sub-jobs like "12345.batch"
                continue
            parts = line.split('|')
            if len(parts) >= 2:
                job_id, state = parts[0], parts[1]
                if state in ['COMPLETED', 'FAILED', 'CANCELLED', 'TIMEOUT']:
                    completed.add(job_id)
                    if state != 'COMPLETED':
                        failed_jobs.append((job_id, state))

        pending_jobs -= completed

        clear_output(wait=True)
        print(f"[{time.strftime('%H:%M:%S')}] Jobs status:")
        print(f"  Completed: {len(job_ids) - len(pending_jobs)}/{len(job_ids)}")
        print(f"  Pending: {len(pending_jobs)}")
        if failed_jobs:
            print(f"  Failed: {failed_jobs}")

        if pending_jobs:
            print()
            print(f"Waiting {poll_interval}s before next check...")
            time.sleep(poll_interval)

    print()
    print('=' * 50)
    if failed_jobs:
        print(f"WARNING: {len(failed_jobs)} jobs failed: {failed_jobs}")
        return False

    print('All jobs completed successfully!')
    return True



# Step 0: Discover sessions and `.raw` files

Discovers `*-good` folders directly under `main_folder_local` and picks one `.raw` per session (prefers `Image_001_001.raw`).


In [ ]:
session_folders_local = sorted([
    p for p in glob.glob(os.path.join(main_folder_local, '*'))
    if os.path.isdir(p) and re.search(r'(^|/)\d+-good$', p)
])
print(f'Found {len(session_folders_local)} session folders.')
for p in session_folders_local:
    print('  -', p)

raw_paths_local = []
for sess in session_folders_local:
    raws = sorted(glob.glob(os.path.join(sess, '**', '*.raw'), recursive=True))
    if not raws:
        print('[WARN] No .raw found under:', sess)
        continue

    preferred = [p for p in raws if os.path.basename(p) == 'Image_001_001.raw']
    pick = preferred[0] if preferred else raws[0]
    if len(raws) > 1:
        print('[WARN] Multiple .raw found under:', sess)
        for p in raws:
            print('    -', p)
        print('    -> using:', pick)
    raw_paths_local.append(pick)

raw_paths_local = sorted(raw_paths_local)
raw_paths = [p.replace('/Volumes/adam-lab', '/ems/elsc-labs/adam-y') for p in raw_paths_local]

print()
print('Final list of .raw file paths (cluster):')
for p in raw_paths:
    print(p)


# Step 1: Concatenate the raw sessions

Builds `output_concat/Image_001_001.raw` on the cluster and copies `Experiment.xml` + `ROIs.xaml` into `output_concat/` so motion correction can run on the concatenated dataset.


In [ ]:
python_script = 'utils/DS_motion_correction_concat.py'
LOG_DIR = f"{PIPELINE_WORKDIR}/logs"
LOG_DIR_LOCAL = LOG_DIR.replace('/ems/elsc-labs/adam-y', '/Volumes/adam-lab')
JOB_NAME = 'DS_motion_correction_concat'

# Ensure output + logs directories exist on the cluster
_, mkdir_err = run_command(f"mkdir -p {LOG_DIR} {output_concat}")
if mkdir_err:
    print(f"[WARN] mkdir returned: {mkdir_err}")

raw_args = ' '.join([f"--raw {p}" for p in raw_paths])
cmd = (
    f"sbatch --job-name {JOB_NAME} --chdir {PIPELINE_WORKDIR} "
    f"--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err "
    f"{sh_script} {PIPELINE_WORKDIR} {python_script} "
    f"--out-dir {output_concat} "
    f"--out-raw-name {concat_raw_name} "
    f"--frames-to-truncate-per-session {frames_to_truncate_per_session} "
    f"--overwrite "
    f"--mc-output-subdir {mc_output_subdir} "
    + (f"--mc-run-tag {mc_run_tag} " if mc_run_tag else "--mc-run-tag '' ")
    + f"--factor {mc_factor[0]} {mc_factor[1]} "
    + ("--skip-roi-detection " if mc_skip_roi_detection else "")
    + raw_args
    + f" --mc-args --niter-rig {int(mc_niter_rig)}"
)

output, error = run_command(cmd)
concat_mc_job_id = None
if error:
    print(f"[ERROR] Failed to submit concat+MC job: {error}")
else:
    print(f"[INFO] Concat+MC job submitted: {output}")
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        concat_mc_job_id = match.group(1)
        print(f"[INFO] Job ID: {concat_mc_job_id}")
        print(f"[INFO] Slurm .out (local): {LOG_DIR_LOCAL}/{JOB_NAME}_{concat_mc_job_id}.out")
        print(f"[INFO] Output concat folder (local): {output_concat_local}")


# Step 2: Submit full-session motion projection and ALI

This step starts from `movReg.tif`, the illumination-corrected movie passed through NoRMCorre. Motion-linked signal is projected independently from every pixel; no manual ROI mask is involved.

ALI then analyzes every frame of the projected movie using the parameters selected in the local 60-second experiments:

- positive-going indicator;
- 50-ms (25-frame at 500 Hz) median residual for event localization;
- coarse threshold 4 SD;
- minimum center separation 8 camera pixels;
- event-assignment radius 5 pixels;
- minimum four assigned events;
- footprint radius 8.5 pixels and smoothing sigma 0.75 pixels.

The SLM polygons are dilated by eight pixels and used only as an automatic search guide. Full-band motion-projected data are used for the saved demixed traces.


In [ ]:
concat_raw_path = os.path.join(output_concat, concat_raw_name)
data_file_paths = [concat_raw_path]
OUTPUT_SUBDIR = mc_output_subdir
RUN_TAG = mc_run_tag

ali_output_folder = os.path.join(
    os.path.dirname(concat_raw_path),
    OUTPUT_SUBDIR,
)
if RUN_TAG:
    ali_output_folder = os.path.join(ali_output_folder, RUN_TAG)

roi_filename = 'Image_001_001_ROIs.tif'
if mc_factor != (1, 1):
    roi_filename = (
        f'Image_001_001_ROIs_ds{mc_factor[0]}x{mc_factor[1]}.tif'
    )
ali_roi_tif = os.path.join(ali_output_folder, roi_filename)

ALI_RESULTS_SUBDIR = (
    'ali_full_coarse4p0_sep8_assign5_minspikes4_'
    'radius8p5_smooth0p75'
)
ali_results_folder = os.path.join(
    ali_output_folder,
    ALI_RESULTS_SUBDIR,
    'result',
)

print('ALI input folder:', ali_output_folder)
print('Automatic SLM guide:', ali_roi_tif)
print('ALI result folder:', ali_results_folder)
print('Manual cell ROIs used: False')


In [ ]:
# Submit one resumable full-session ALI job.
LOG_DIR = f"{PIPELINE_WORKDIR}/logs"
LOG_DIR_LOCAL = LOG_DIR.replace(
    '/ems/elsc-labs/adam-y',
    '/Volumes/adam-lab',
)
JOB_NAME = 'full_concat_ali'
ALI_MEM_GB = 128
ALI_CPUS = 8
ALI_TIME_LIMIT = '7-00:00:00'
OVERWRITE_MOTION_PROJECTION = False
OVERWRITE_ALI = False

python_script = 'utils/run_ali_concat.py'
pyali_root = f'{PIPELINE_WORKDIR}/third_party/pyALI'

_, mkdir_err = run_command(f"mkdir -p {LOG_DIR}")
if mkdir_err:
    print(f"[WARN] Failed to create log dir on cluster: {mkdir_err}")

dependency = (
    f'--dependency=afterok:{concat_mc_job_id} '
    if globals().get('concat_mc_job_id')
    else ''
)
cmd = (
    f"sbatch {dependency}--job-name {JOB_NAME} "
    f"--cpus-per-task {ALI_CPUS} --mem={ALI_MEM_GB}G "
    f"--time {ALI_TIME_LIMIT} --chdir {PIPELINE_WORKDIR} "
    f"--output {LOG_DIR}/%x_%j.out --error {LOG_DIR}/%x_%j.err "
    f"{sh_script} {PIPELINE_WORKDIR} {python_script} "
    f"{ali_output_folder} "
    f"--source-name movReg.tif "
    f"--projection-name movReg_motion_projected.tif "
    f"--projection-summary-name movReg_motion_projection_summary.npz "
    f"--roi-tif {ali_roi_tif} "
    f"--frame-rate 500 "
    f"--pyali-root {pyali_root} "
    f"--results-subdir {ALI_RESULTS_SUBDIR} "
    f"--motion-trend-seconds 4.0 "
    f"--motion-jitter-seconds 0.010 "
    f"--motion-epoch-seconds 10.0 "
    f"--median-window-ms 50 "
    f"--search-dilation-px 8 "
    f"--coarse-threshold-sd 4.0 "
    f"--minimum-segment-voxels 5 "
    f"--spatial-sigma-px 2.5 "
    f"--spatial-radius-px 5 "
    f"--ali-map-sigma 1.0 "
    f"--ali-map-radius 3 "
    f"--ali-peak-threshold 0.30 "
    f"--center-separation-px 8.0 "
    f"--assignment-radius-px 5.0 "
    f"--minimum-cluster-spikes 4 "
    f"--footprint-radius-px 8.5 "
    f"--footprint-smoothing-sigma-px 0.75 "
    f"--maximum-coarse-events 0 "
)
if OVERWRITE_MOTION_PROJECTION:
    cmd += ' --overwrite-motion-projection'
if OVERWRITE_ALI:
    cmd += ' --overwrite-ali'

output, error = run_command(cmd)
ali_job_id = None
if error:
    print(f"[ERROR] Failed to submit full-session ALI job: {error}")
else:
    print(f"[INFO] Full-session ALI job submitted: {output}")
    match = re.search(r'Submitted batch job (\d+)', output)
    if match:
        ali_job_id = match.group(1)
        print(f"[INFO] Job ID: {ali_job_id}")
        print(
            f"[INFO] Slurm .out (local): "
            f"{LOG_DIR_LOCAL}/{JOB_NAME}_{ali_job_id}.out"
        )
        print(
            f"[INFO] Slurm .err (local): "
            f"{LOG_DIR_LOCAL}/{JOB_NAME}_{ali_job_id}.err"
        )
        print('[INFO] Submission is non-blocking; use the next cell to monitor.')


In [ ]:
# Optional: monitor the ALI job from this notebook.
if globals().get('ali_job_id'):
    wait_for_jobs([ali_job_id], poll_interval=30)
else:
    print('[INFO] No ali_job_id found. Submit Step 2 first.')


# Step 3: Inspect the completed automatic ALI result

The cluster job writes:

- `pyali_results.npz`: footprints, full-band traces, detection traces, event locations, and masks;
- `pyali_metadata.json`: parameters and provenance;
- `ali_run_summary.json`: compact completion summary;
- `ali_qc_first60s.png`: static first-minute QC;
- `ali_dashboard_bundle.pickle`: input for the local Dash viewer.

The large projected and 50-ms residual TIFFs remain beside the cluster output for restartability.


In [ ]:
ali_results_folder_local = ali_results_folder.replace(
    '/ems/elsc-labs/adam-y',
    '/Volumes/adam-lab',
)
ali_summary_path_local = os.path.join(
    ali_results_folder_local,
    'ali_run_summary.json',
)
ali_qc_path_local = os.path.join(
    ali_results_folder_local,
    'ali_qc_first60s.png',
)
ali_dashboard_path_local = os.path.join(
    ali_results_folder_local,
    'ali_dashboard_bundle.pickle',
)

if os.path.exists(ali_summary_path_local):
    import json

    with open(ali_summary_path_local, 'r') as handle:
        ali_summary = json.load(handle)
    print(json.dumps(ali_summary, indent=2))
else:
    print('[INFO] ALI summary is not available yet:', ali_summary_path_local)

if os.path.exists(ali_qc_path_local):
    from IPython.display import Image, display

    display(Image(filename=ali_qc_path_local))
else:
    print('[INFO] ALI QC image is not available yet:', ali_qc_path_local)


In [ ]:
# Launch the existing local dashboard with the automatic ALI bundle.
import subprocess

if not os.path.exists(ali_dashboard_path_local):
    raise FileNotFoundError(ali_dashboard_path_local)

app_script = os.path.join(
    PIPELINE_WORKDIR.replace(
        '/ems/elsc-labs/adam-y',
        '/Volumes/adam-lab',
    ),
    'dash_overview_app',
    'app.py',
)
args = ['python', app_script, ali_dashboard_path_local]
print('Running command:', ' '.join(args))
subprocess.run(args, check=True)


# Operational notes

- Re-running with overwrite flags disabled reuses compatible motion-projection, high-pass, and coarse-event caches.
- A whole movie is processed as one ALI dataset; it is not demixed independently by temporal chunk and concatenated afterward.
- Temporal chunks in the utility bound I/O and memory only. Clustering uses the complete event set, and trace extraction spans the complete projected movie.
- `--maximum-coarse-events 0` intentionally removes the 60-second notebook's 5,000-event safety ceiling.
- If the job is preempted while writing a large TIFF, remove only that incomplete product or rerun the corresponding stage with its explicit overwrite flag.
